# PBR Data Preprocessing — Train V2

Split the raw 53k-player PBR scrape into **pitcher** and **hitter** CSVs with cleaned numeric values.

**Source:** `backend/data/rescraped/pbr_all_players.csv` (untouched)
**Outputs:**
- `pitchers/pitchers_cleaned.csv`
- `hitters/hitters_cleaned.csv`

### Cleaning steps
1. Split by primary position (RHP/LHP = pitchers, all others = hitters)
2. Resolve range values (e.g. "80 - 84" -> 82.0) for velo ranges and pop times
3. Strip commas from numbers ("3,409.1" -> 3409.1)
4. NaN unrealistic outliers (height, weight, stat values)
5. Parse age string ("17yr 5mo") to numeric years
6. Remap ambiguous positions ("-", "MIF", "UTILITY", etc.)

In [16]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

# Paths
RAW_CSV = Path("../../../data/rescraped/pbr_all_players_patched.csv")
PITCHERS_OUT = Path("pitchers/pitchers_cleaned.csv")
HITTERS_OUT = Path("hitters/hitters_cleaned.csv")

os.makedirs("pitchers", exist_ok=True)
os.makedirs("hitters", exist_ok=True)

df = pd.read_csv(RAW_CSV, low_memory=False)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
df.head(3)

Loaded 53,085 rows, 70 columns


,name,link,player_state,high_school,class,primary_position,commitment,commitment_date,age,positions,...,c_velo,c_velo_date,pop_time,pop_time_date,player_region,conference,division,college_location,committment_group,scraped_at
0,Koleson Abbott,https://www.prepbaseballreport.com/profiles/IN...,IN,Leo Junior/Senior,2027,1B,Grace College,NaN,17yr 5mo,"1B,3B",...,NaN,NaN,NaN,NaN,Midwest,Crossroads,NAIA,"WINONA LAKE, IN",Non D1,2026-04-17T21:11:27.791288
1,Enmanuel Acevedo,https://www.prepbaseballreport.com/profiles/NY...,NY,Poly Prep,2027,RHP,Virginia,11/12/25,18yr,RHP,...,NaN,NaN,NaN,NaN,Northeast,Atlantic Coast Conference,NCAA I,"CHARLOTTESVILLE, VA",P4,2026-04-17T21:11:36.387287
2,Riley Ackerman,https://www.prepbaseballreport.com/profiles/IN...,IN,Crown Point,2027,LHP,Northwestern,10/21/25,17yr 4mo,LHP,...,NaN,NaN,NaN,NaN,Midwest,BIG 10,NCAA I,"EVANSTON, IL",Non P4 D1,2026-04-17T21:11:45.340374


## 1. Data Overview

In [17]:
# Position distribution
print("=== Primary Position Distribution ===")
print(df["primary_position"].value_counts().to_string())
print(f"\nTotal: {len(df):,}")

print("\n=== Commitment Group Distribution ===")
print(df["committment_group"].value_counts().to_string())

print("\n=== Class Year Distribution ===")
print(df["class"].value_counts().sort_index().to_string())

=== Primary Position Distribution ===
primary_position
RHP        14544
OF          9188
SS          7839
C           6844
LHP         4907
3B          3637
1B          3041
2B          2200
-            706
MIF           96
UTILITY       58
CF            18
DH             2
IF             2
RF             2
LF             1

Total: 53,085

=== Commitment Group Distribution ===
committment_group
Non D1       34816
Non P4 D1    11000
Unknown       4163
P4            3106

=== Class Year Distribution ===
class
2022    10548
2023    10720
2024    10404
2025    10787
2026     9434
2027     1192


In [18]:
# Non-null counts for key fields
key_cols = [
    "height", "weight", "throwing_hand", "hitting_handedness",
    "fastball_velo_max", "fastball_velo_range", "fastball_spin",
    "exit_velo_max", "exit_velo_avg", "bat_speed_max", "hand_speed_max",
    "sixty_time", "inf_velo", "of_velo", "c_velo", "pop_time",
    "distance_max", "sweet_spot_p", "hard_hit_p",
    "commitment_date", "committment_group",
]
coverage = df[key_cols].notna().sum().rename("non_null")
coverage = pd.DataFrame(coverage)
coverage["pct"] = (coverage["non_null"] / len(df) * 100).round(1)
coverage

,non_null,pct
height,52008,98.0
weight,48121,90.6
throwing_hand,51257,96.6
hitting_handedness,51257,96.6
fastball_velo_max,30414,57.3
fastball_velo_range,31447,59.2
fastball_spin,23514,44.3
exit_velo_max,37786,71.2
exit_velo_avg,33058,62.3
bat_speed_max,30170,56.8


## 2. Cleaning Functions

All cleaning is applied to the split DataFrames, never the raw CSV.

In [19]:
def avg_range(val):
    """Convert range string '80 - 84' to midpoint 82.0. Pass through single numbers."""
    if pd.isna(val) or val == "":
        return np.nan
    val = str(val).strip()
    # Range pattern: "80 - 84" or "94.8 - 96.2"
    m = re.match(r"^([\d.]+)\s*-\s*([\d.]+)$", val)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    # Single number (possibly with comma)
    try:
        return float(val.replace(",", ""))
    except ValueError:
        return np.nan


def clean_numeric(val):
    """Convert string to float, stripping commas. Returns NaN for non-numeric."""
    if pd.isna(val) or val == "":
        return np.nan
    try:
        return float(str(val).replace(",", "").strip())
    except ValueError:
        return np.nan


def nan_outliers(series, low, high):
    """Replace values outside [low, high] with NaN."""
    return series.where(series.between(low, high))


# Sanity checks
assert avg_range("80 - 84") == 82.0
assert avg_range("94.8 - 96.2") == 95.5
assert avg_range("1.87 - 1.97") == 1.92
assert avg_range("68 - 68") == 68.0
assert clean_numeric("3,409.1") == 3409.1
assert pd.isna(clean_numeric(""))
print("All cleaning functions pass.")

All cleaning functions pass.


In [20]:
def apply_shared_cleaning(data: pd.DataFrame) -> pd.DataFrame:
    """Apply all cleaning steps shared between pitchers and hitters."""
    df = data.copy()

    # --- Height: valid range 54-83 inches (4'6" to 6'11") ---
    df["height"] = pd.to_numeric(df["height"], errors="coerce")
    before_h = df["height"].notna().sum()
    df["height"] = nan_outliers(df["height"], 54, 83)
    print(f"Height: {before_h - df['height'].notna().sum()} outliers removed")

    # --- Weight: valid range 100-290 lbs ---
    df["weight"] = pd.to_numeric(df["weight"], errors="coerce")
    before_w = df["weight"].notna().sum()
    df["weight"] = nan_outliers(df["weight"], 100, 290)
    print(f"Weight: {before_w - df['weight'].notna().sum()} outliers removed")

    # --- Range columns -> midpoint ---
    range_cols = [
        "fastball_velo_range", "changeup_velo_range",
        "curveball_velo_range", "slider_velo_range", "pop_time",
    ]
    for col in range_cols:
        if col in df.columns:
            df[col] = df[col].apply(avg_range)

    # --- All other numeric stat columns -> clean_numeric ---
    numeric_stat_cols = [
        "fastball_velo_max", "fastball_spin",
        "changeup_spin", "curveball_spin", "slider_spin",
        "exit_velo_max", "exit_velo_avg", "distance_max",
        "sweet_spot_p", "hand_speed_max", "bat_speed_max",
        "rot_acc_max", "hard_hit_p",
        "sixty_time", "thirty_time", "ten_yard_time", "run_speed_max",
        "inf_velo", "of_velo", "c_velo",
    ]
    for col in numeric_stat_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_numeric)

    # --- Stat outlier bounds ---
    # These are generous bounds; anything outside is clearly a scraping artifact
    outlier_bounds = {
        "fastball_velo_max": (40, 105),
        "fastball_spin": (800, 3200),
        "exit_velo_max": (40, 130),
        "exit_velo_avg": (40, 115),
        "distance_max": (50, 450),
        "sweet_spot_p": (0, 100),
        "hand_speed_max": (10, 50),
        "bat_speed_max": (40, 100),
        "rot_acc_max": (0, 55),
        "hard_hit_p": (0, 100),
        "sixty_time": (5.5, 9.5),
        "thirty_time": (3.0, 5.5),
        "ten_yard_time": (1.0, 3.0),
        "run_speed_max": (5, 30),
        "inf_velo": (40, 105),
        "of_velo": (40, 105),
        "c_velo": (40, 95),
        "pop_time": (1.5, 3.0),
        "fastball_velo_range": (40, 105),
        "changeup_velo_range": (30, 100),
        "curveball_velo_range": (30, 100),
        "slider_velo_range": (30, 100),
        "changeup_spin": (500, 3000),
        "curveball_spin": (500, 4000),
        "slider_spin": (500, 3500),
    }
    total_outliers = 0
    for col, (lo, hi) in outlier_bounds.items():
        if col in df.columns:
            before = df[col].notna().sum()
            df[col] = nan_outliers(df[col], lo, hi)
            removed = before - df[col].notna().sum()
            if removed > 0:
                total_outliers += removed
                print(f"  {col}: {removed} outliers removed (bounds [{lo}, {hi}])")
    print(f"Total stat outliers removed: {total_outliers}")

    return df


print("Shared cleaning function defined.")

Shared cleaning function defined.


## 3. Position Remapping & Split

**Pitcher:** primary_position in (RHP, LHP)
**Hitter:** everything else — OF, SS, C, 3B, 1B, 2B, and minor categories

Players with position "-" (706): use their `positions` field if available to reclassify, otherwise drop.
Minor categories (MIF, UTILITY, CF, DH, IF, RF, LF) are mapped to standard positions.

In [21]:
# Remap minor position labels to standard ones
POSITION_REMAP = {
    "CF": "OF", "RF": "OF", "LF": "OF",   # outfield variants -> OF
    "MIF": "SS",                             # middle infield -> SS
    "IF": "SS",                              # generic infield -> SS
    "DH": "1B",                              # DH -> 1B (closest role)
    "UTILITY": "OF",                         # utility -> OF (most common utility slot)
}

PITCHER_POSITIONS = {"RHP", "LHP"}
HITTER_POSITIONS = {"OF", "SS", "C", "3B", "1B", "2B"}

def resolve_position(row):
    """Resolve primary_position, using positions field as fallback for '-'."""
    pos = str(row["primary_position"]).strip()

    # Already a known position
    if pos in PITCHER_POSITIONS or pos in HITTER_POSITIONS:
        return pos
    if pos in POSITION_REMAP:
        return POSITION_REMAP[pos]

    # Position is "-" or unknown: try the positions field
    positions_str = str(row.get("positions", "")).strip()
    if positions_str:
        for p in positions_str.split(","):
            p = p.strip()
            if p in PITCHER_POSITIONS or p in HITTER_POSITIONS:
                return p
            if p in POSITION_REMAP:
                return POSITION_REMAP[p]

    return None  # can't resolve -> will be dropped


df["resolved_position"] = df.apply(resolve_position, axis=1)

resolved = df["resolved_position"].value_counts(dropna=False)
print("=== Resolved Position Distribution ===")
print(resolved.to_string())

unresolved = df["resolved_position"].isna().sum()
print(f"\nUnresolvable (will be dropped): {unresolved}")

=== Resolved Position Distribution ===
resolved_position
RHP     14593
OF       9281
SS       7951
C        6850
LHP      4913
3B       3644
1B       3047
2B       2226
None      580

Unresolvable (will be dropped): 580


In [22]:
# Split into pitchers and hitters
df_resolved = df[df["resolved_position"].notna()].copy()

pitchers_raw = df_resolved[df_resolved["resolved_position"].isin(PITCHER_POSITIONS)].copy()
hitters_raw = df_resolved[df_resolved["resolved_position"].isin(HITTER_POSITIONS)].copy()

print(f"Pitchers: {len(pitchers_raw):,}")
print(f"Hitters:  {len(hitters_raw):,}")
print(f"Dropped:  {len(df) - len(df_resolved):,} (unresolvable position)")
print(f"Total:    {len(pitchers_raw) + len(hitters_raw):,}")

Pitchers: 19,506
Hitters:  32,999
Dropped:  580 (unresolvable position)
Total:    52,505


## 4. Clean Pitcher Data

In [23]:
pitchers = apply_shared_cleaning(pitchers_raw)

# Pitcher-relevant columns to keep
# NOTE: age excluded — PBR age is computed from today's date,
# so it just reflects class year (2022 class = ~22yr, 2027 = ~17yr). Not useful for ML.
pitcher_cols = [
    # Identity / bio
    "name", "link", "player_state", "high_school", "class",
    "resolved_position", "positions", "commitment", "commitment_date",
    "height", "weight", "throwing_hand", "hitting_handedness",
    # Pitching stats (value + date for each)
    "fastball_velo_max", "fastball_velo_max_date",
    "fastball_velo_range", "fastball_velo_range_date",
    "fastball_spin", "fastball_spin_date",
    "changeup_velo_range", "changeup_velo_range_date",
    "changeup_spin", "changeup_spin_date",
    "curveball_velo_range", "curveball_velo_range_date",
    "curveball_spin", "curveball_spin_date",
    "slider_velo_range", "slider_velo_range_date",
    "slider_spin", "slider_spin_date",
    # Running (value + date)
    "sixty_time", "sixty_time_date",
    "thirty_time", "thirty_time_date",
    "ten_yard_time", "ten_yard_time_date",
    "run_speed_max", "run_speed_max_date",
    # Classification
    "player_region", "conference", "division", "college_location",
    "committment_group",
]
# Only keep cols that exist
pitcher_cols = [c for c in pitcher_cols if c in pitchers.columns]
pitchers = pitchers[pitcher_cols]

print(f"Pitchers: {len(pitchers):,} rows, {len(pitchers.columns)} columns")
print(f"\n=== Pitcher Stat Coverage ===")
pitcher_stat_cols = [
    "fastball_velo_max", "fastball_velo_range", "fastball_spin",
    "changeup_velo_range", "changeup_spin",
    "curveball_velo_range", "curveball_spin",
    "slider_velo_range", "slider_spin",
    "sixty_time",
]
for col in pitcher_stat_cols:
    n = pitchers[col].notna().sum()
    print(f"  {col}: {n:,} ({100*n/len(pitchers):.0f}%)")

print(f"\n=== Pitcher Commitment Group ===")
print(pitchers["committment_group"].value_counts().to_string())

Height: 45 outliers removed
Weight: 15 outliers removed
  fastball_velo_max: 1 outliers removed (bounds [40, 105])
  exit_velo_max: 3 outliers removed (bounds [40, 130])
  exit_velo_avg: 3 outliers removed (bounds [40, 115])
  distance_max: 5 outliers removed (bounds [50, 450])
  sixty_time: 18 outliers removed (bounds [5.5, 9.5])
  thirty_time: 16 outliers removed (bounds [3.0, 5.5])
  ten_yard_time: 8 outliers removed (bounds [1.0, 3.0])
  run_speed_max: 16 outliers removed (bounds [5, 30])
  pop_time: 3 outliers removed (bounds [1.5, 3.0])
  fastball_velo_range: 2 outliers removed (bounds [40, 105])
  changeup_velo_range: 1 outliers removed (bounds [30, 100])
  curveball_velo_range: 1 outliers removed (bounds [30, 100])
  slider_velo_range: 1 outliers removed (bounds [30, 100])
  changeup_spin: 3 outliers removed (bounds [500, 3000])
  curveball_spin: 1 outliers removed (bounds [500, 4000])
Total stat outliers removed: 82
Pitchers: 19,506 rows, 44 columns

=== Pitcher Stat Coverage 

In [24]:
# Quick sanity check on cleaned pitcher stats
pitcher_check_cols = ["fastball_velo_max", "fastball_velo_range", "fastball_spin", "sixty_time", "height", "weight"]
pitchers[pitcher_check_cols].describe().round(1)

,fastball_velo_max,fastball_velo_range,fastball_spin,sixty_time,height,weight
count,16518.0,17039.0,14073.0,9657.0,19154.0,18043.0
mean,85.4,83.7,2080.4,7.5,73.2,185.4
std,4.9,4.9,203.2,0.4,2.4,21.3
min,59.0,43.0,1245.0,6.3,54.0,100.0
25%,82.2,80.6,1951.0,7.2,72.0,170.0
50%,86.0,84.0,2087.0,7.4,73.0,185.0
75%,89.0,87.0,2216.0,7.7,75.0,200.0
max,102.0,99.5,3129.0,9.5,83.0,290.0


## 5. Clean Hitter Data

In [25]:
hitters = apply_shared_cleaning(hitters_raw)

# Hitter-relevant columns to keep
# NOTE: age excluded — see pitcher cell for reasoning
# NOTE: pitching stats excluded — this is for hitter models only
hitter_cols = [
    # Identity / bio
    "name", "link", "player_state", "high_school", "class",
    "resolved_position", "positions", "commitment", "commitment_date",
    "height", "weight", "throwing_hand", "hitting_handedness",
    # Hitting / power stats (value + date for each)
    "exit_velo_max", "exit_velo_max_date",
    "exit_velo_avg", "exit_velo_avg_date",
    "distance_max", "distance_max_date",
    "sweet_spot_p", "sweet_spot_p_date",
    "hand_speed_max", "hand_speed_max_date",
    "bat_speed_max", "bat_speed_max_date",
    "rot_acc_max", "rot_acc_max_date",
    "hard_hit_p", "hard_hit_p_date",
    # Running (value + date)
    "sixty_time", "sixty_time_date",
    "thirty_time", "thirty_time_date",
    "ten_yard_time", "ten_yard_time_date",
    "run_speed_max", "run_speed_max_date",
    # Defense (value + date)
    "inf_velo", "inf_velo_date",
    "of_velo", "of_velo_date",
    "c_velo", "c_velo_date",
    "pop_time", "pop_time_date",
    # Classification
    "player_region", "conference", "division", "college_location",
    "committment_group",
]
hitter_cols = [c for c in hitter_cols if c in hitters.columns]
hitters = hitters[hitter_cols]

print(f"Hitters: {len(hitters):,} rows, {len(hitters.columns)} columns")
print(f"\n=== Hitter Stat Coverage ===")
hitter_stat_cols = [
    "exit_velo_max", "exit_velo_avg", "bat_speed_max", "hand_speed_max",
    "distance_max", "sweet_spot_p", "hard_hit_p",
    "sixty_time", "inf_velo", "of_velo", "c_velo", "pop_time",
]
for col in hitter_stat_cols:
    n = hitters[col].notna().sum()
    print(f"  {col}: {n:,} ({100*n/len(hitters):.0f}%)")

print(f"\n=== Hitter Position Breakdown ===")
print(hitters["resolved_position"].value_counts().to_string())

print(f"\n=== Hitter Commitment Group ===")
print(hitters["committment_group"].value_counts().to_string())

Height: 112 outliers removed
Weight: 20 outliers removed
  fastball_velo_max: 6 outliers removed (bounds [40, 105])
  exit_velo_avg: 1 outliers removed (bounds [40, 115])
  distance_max: 3 outliers removed (bounds [50, 450])
  sixty_time: 14 outliers removed (bounds [5.5, 9.5])
  thirty_time: 54 outliers removed (bounds [3.0, 5.5])
  ten_yard_time: 10 outliers removed (bounds [1.0, 3.0])
  run_speed_max: 56 outliers removed (bounds [5, 30])
  pop_time: 8 outliers removed (bounds [1.5, 3.0])
  fastball_velo_range: 1 outliers removed (bounds [40, 105])
  changeup_spin: 2 outliers removed (bounds [500, 3000])
Total stat outliers removed: 155
Hitters: 32,999 rows, 50 columns

=== Hitter Stat Coverage ===
  exit_velo_max: 27,497 (83%)
  exit_velo_avg: 24,604 (75%)
  bat_speed_max: 22,934 (69%)
  hand_speed_max: 22,934 (69%)
  distance_max: 24,605 (75%)
  sweet_spot_p: 24,552 (74%)
  hard_hit_p: 16,797 (51%)
  sixty_time: 26,086 (79%)
  inf_velo: 17,966 (54%)
  of_velo: 10,591 (32%)
  c_velo

In [26]:
# Quick sanity check on cleaned hitter stats
hitter_check_cols = ["exit_velo_max", "exit_velo_avg", "bat_speed_max", "sixty_time", "height", "weight"]
hitters[hitter_check_cols].describe().round(1)

,exit_velo_max,exit_velo_avg,bat_speed_max,sixty_time,height,weight
count,27497.0,24604.0,22934.0,26086.0,32517.0,29886.0
mean,92.0,84.3,73.1,7.2,71.6,179.7
std,6.3,6.4,5.7,0.4,2.4,21.0
min,40.4,40.2,49.8,6.0,56.0,100.0
25%,88.2,80.4,69.3,7.0,70.0,165.0
50%,92.3,84.8,72.9,7.2,72.0,180.0
75%,96.1,88.7,76.6,7.5,73.0,190.0
max,129.4,110.8,90.0,9.5,83.0,290.0


## 6. Final Inspection — Catcher Sub-group

Catchers are in the hitters CSV but have unique stats (c_velo, pop_time). Quick check that their data looks right.

In [27]:
catchers = hitters[hitters["resolved_position"] == "C"]
print(f"Catchers: {len(catchers):,}")

catcher_stats = ["c_velo", "pop_time", "exit_velo_max", "bat_speed_max", "sixty_time", "inf_velo"]
for col in catcher_stats:
    n = catchers[col].notna().sum()
    print(f"  {col}: {n:,} ({100*n/len(catchers):.0f}%)")

print(f"\nCatcher commitment group:")
print(catchers["committment_group"].value_counts().to_string())

print(f"\nCatcher pop_time + c_velo stats:")
catchers[["c_velo", "pop_time"]].describe().round(2)

Catchers: 6,850
  c_velo: 5,718 (83%)
  pop_time: 5,677 (83%)
  exit_velo_max: 5,804 (85%)
  bat_speed_max: 4,899 (72%)
  sixty_time: 5,509 (80%)
  inf_velo: 2,084 (30%)

Catcher commitment group:
committment_group
Non D1       4755
Non P4 D1    1226
Unknown       592
P4            277

Catcher pop_time + c_velo stats:


,c_velo,pop_time
count,5718.00,5677.00
mean,75.49,2.06
std,4.24,0.12
min,54.00,1.64
25%,73.00,1.98
50%,76.00,2.04
75%,78.00,2.13
max,92.00,2.98


## 7. P4 vs Non-P4 D1 vs Non-D1 — Stat Comparison

Quick look at whether the cleaned stats show meaningful separation between commitment tiers.

In [28]:
# Pitcher stat means by commitment group
pitcher_compare_cols = ["fastball_velo_max", "fastball_velo_range", "fastball_spin", "sixty_time", "height", "weight"]
print("=== Pitcher Means by Commitment Group ===")
pitcher_groups = pitchers[pitchers["committment_group"].isin(["P4", "Non P4 D1", "Non D1"])]
pitcher_groups.groupby("committment_group")[pitcher_compare_cols].mean().round(1).reindex(["P4", "Non P4 D1", "Non D1"])

=== Pitcher Means by Commitment Group ===


,fastball_velo_max,fastball_velo_range,fastball_spin,sixty_time,height,weight
committment_group,,,,,,
P4,91.1,89.0,2209.1,7.4,74.6,194.5
Non P4 D1,88.3,86.4,2149.4,7.4,73.8,188.8
Non D1,83.8,82.2,2043.5,7.5,72.9,183.4


In [29]:
# Hitter stat means by commitment group
hitter_compare_cols = ["exit_velo_max", "exit_velo_avg", "bat_speed_max", "distance_max", "sixty_time", "height", "weight"]
print("=== Hitter Means by Commitment Group ===")
hitter_groups = hitters[hitters["committment_group"].isin(["P4", "Non P4 D1", "Non D1"])]
hitter_groups.groupby("committment_group")[hitter_compare_cols].mean().round(1).reindex(["P4", "Non P4 D1", "Non D1"])

=== Hitter Means by Commitment Group ===


,exit_velo_max,exit_velo_avg,bat_speed_max,distance_max,sixty_time,height,weight
committment_group,,,,,,,
P4,97.0,89.6,76.8,357.8,7.0,72.8,189.2
Non P4 D1,94.5,87.1,75.1,342.2,7.1,72.1,183.5
Non D1,91.2,83.5,72.4,319.6,7.3,71.3,178.5


In [30]:
# Catcher stat means by commitment group
catcher_compare_cols = ["c_velo", "pop_time", "exit_velo_max", "bat_speed_max", "sixty_time", "height", "weight"]
print("=== Catcher Means by Commitment Group ===")
catcher_groups = catchers[catchers["committment_group"].isin(["P4", "Non P4 D1", "Non D1"])]
catcher_groups.groupby("committment_group")[catcher_compare_cols].mean().round(2).reindex(["P4", "Non P4 D1", "Non D1"])

=== Catcher Means by Commitment Group ===


,c_velo,pop_time,exit_velo_max,bat_speed_max,sixty_time,height,weight
committment_group,,,,,,,
P4,79.67,1.97,97.32,76.74,7.17,72.58,194.87
Non P4 D1,77.98,2.00,94.76,75.11,7.26,71.91,189.99
Non D1,74.75,2.08,91.10,72.38,7.44,71.13,182.99


## 8. Save Cleaned CSVs

In [32]:
# Save cleaned CSVs
pitchers.to_csv(PITCHERS_OUT, index=False)
hitters.to_csv(HITTERS_OUT, index=False)

print(f"Saved pitchers: {PITCHERS_OUT} ({len(pitchers):,} rows)")
print(f"Saved hitters:  {HITTERS_OUT} ({len(hitters):,} rows)")

# Final summary
print(f"\n{'='*50}")
print(f"PREPROCESSING COMPLETE")
print(f"{'='*50}")
print(f"Raw input:     {len(df):,} players")
print(f"Dropped:       {len(df) - len(pitchers) - len(hitters):,} (unresolvable position)")
print(f"Pitchers:      {len(pitchers):,}")
print(f"  RHP: {len(pitchers[pitchers['resolved_position']=='RHP']):,}")
print(f"  LHP: {len(pitchers[pitchers['resolved_position']=='LHP']):,}")
print(f"Hitters:       {len(hitters):,}")
print(f"  OF:  {len(hitters[hitters['resolved_position']=='OF']):,}")
print(f"  SS:  {len(hitters[hitters['resolved_position']=='SS']):,}")
print(f"  C:   {len(hitters[hitters['resolved_position']=='C']):,}")
print(f"  3B:  {len(hitters[hitters['resolved_position']=='3B']):,}")
print(f"  1B:  {len(hitters[hitters['resolved_position']=='1B']):,}")
print(f"  2B:  {len(hitters[hitters['resolved_position']=='2B']):,}")

Saved pitchers: pitchers/pitchers_cleaned.csv (19,506 rows)
Saved hitters:  hitters/hitters_cleaned.csv (32,999 rows)

PREPROCESSING COMPLETE
Raw input:     53,085 players
Dropped:       580 (unresolvable position)
Pitchers:      19,506
  RHP: 14,593
  LHP: 4,913
Hitters:       32,999
  OF:  9,281
  SS:  7,951
  C:   6,850
  3B:  3,644
  1B:  3,047
  2B:  2,226
